# Python for MATLAB users · 4 · Functions and idioms

The constructs that make Python code look unfamiliar to a MATLAB reader.

**Run this cell first** (Shift+Enter). It downloads the project and installs it
into the Python of this notebook. Every notebook runs on its own machine, so every
notebook starts with this cell, and needs it again when Colab gives it a new machine.

In [ ]:
!test -d /content/pdm-workshop || git clone -q --depth 1 https://github.com/alessandro-rubin-sate/pdm-workshop.git /content/pdm-workshop
%cd /content/pdm-workshop
%pip install -q .

### Idiom 6 - *args and **kwargs: any number of arguments.

`idioms/06_args_kwargs.py`

```
MATLAB:  varargin, and name-value arguments
```

In [ ]:
import math

In [ ]:
# In a definition, the stars collect what is left over
def report(machine, *args, **kwargs):
    print("machine:", machine)
    print("args:   ", args)
    print("kwargs: ", kwargs)


report("M07", 48, 5, unit="h")
report("M03")

In [ ]:
# In a call, the stars spread a sequence or a dictionary into arguments
point = (3, 4)
print("math.hypot(*point):", math.hypot(*point))

options = {"sep": " | ", "end": " <\n"}
print("M01", "M03", **options)

In [ ]:
# Where you will meet them: a wrapper that passes everything on
def loud(function, *args, **kwargs):
    print(f"calling {function.__name__} with {args} and {kwargs}")
    return function(*args, **kwargs)


print(loud(round, 3.14159, ndigits=2))

In [ ]:
# Related: any argument can be given by name, and a lone * makes names compulsory
def window(signal, width=256, *, overlap=0):
    return f"{len(signal)} samples, width {width}, overlap {overlap}"


print(window([0] * 1000))
print(window([0] * 1000, width=128, overlap=64))
try:
    window([0] * 1000, 128, 64)
except TypeError as error:
    print("TypeError:", error)

### Idiom 7 - Functions are values: key= and lambda.

`idioms/07_functions_as_values.py`

```
MATLAB:  function handles, @(x) x.^2
```

In [ ]:
lives = {"M01": 466, "M03": 272, "M08": 582}

In [ ]:
# sorted, min and max take a function that says what to compare
print("by name: ", sorted(lives))
print("by hours:", sorted(lives, key=lives.get))
print("longest: ", max(lives, key=lives.get))

In [ ]:
# lambda writes a small function in place
print("longest first:", sorted(lives, key=lambda m: -lives[m]))

In [ ]:
# The same with a named function: clearer as soon as it grows
def hours_of(machine):
    return lives[machine]


print("by hours:", sorted(lives, key=hours_of))

In [ ]:
# any and all, fed by a comprehension without brackets
print("any above 500:", any(h > 500 for h in lives.values()))
print("all above 300:", all(h > 300 for h in lives.values()))

In [ ]:
# A function stored in a variable, or in a dictionary
transform = abs
print("transform(-3):", transform(-3))

features = {"lowest": min, "highest": max}
for name, function in features.items():
    print(name, function(lives.values()))

### Idiom 8 - f-strings: values inside text.

`idioms/08_fstrings.py`

```
MATLAB:  sprintf("%s: RMS %.2f over %d windows", machine, rms, n)
```

In [ ]:
machine, rms, n = "M07", 1.4826, 12

In [ ]:
# Values go in braces; the format goes after a colon
print(f"{machine}: RMS {rms:.2f} over {n} windows")

In [ ]:
# An equals sign prints the name too: handy while debugging
print(f"{rms=}")
print(f"{n=}, {machine=}")

In [ ]:
# Padding and alignment
print(f"{n:04d}")
print(f"[{machine:>8}]")

In [ ]:
# Joining and splitting
print(", ".join(["M01", "M03", "M08"]))
print("M01,M03,M08".split(","))

In [ ]:
# Strings are never modified in place: every method returns a new one
name = "report_M07.csv"
print(name.removesuffix(".csv"))
print(name.upper())
print(name)

In [ ]:
# Single and double quotes mean the same thing
print("M07" == "M07")

### Idiom 9 - Truthiness, None, and one-line choices.

`idioms/09_truthiness.py`

In [ ]:
import numpy as np

machines = ["M01", "M07"]

In [ ]:
# Empty containers, 0, None and "" count as False
windows = []
if not windows:
    print("nothing to do")

for value in [[], [0], {}, "", "0", 0, 0.0, None]:
    print(f"{value!r:>6} counts as {bool(value)}")

In [ ]:
# None means "no value"; compare with `is`
result = None
if result is None:
    result = 42
print("result:", result)

In [ ]:
# `or` gives the first value that is not empty
user_width = None
width = user_width or 512
print("width:", width)

In [ ]:
# ...which also replaces a legitimate 0: predict this one
threshold = 0
effective = threshold or 4.0
print("threshold 0 becomes:", effective)
effective = 4.0 if threshold is None else threshold
print("with `is None`:    ", effective)

In [ ]:
# A choice in one expression
z = 5.2
label = "alarm" if z > 4 else "ok"
print("label:", label)

In [ ]:
# Membership and chained comparisons
print('"M07" in machines:', "M07" in machines)
print("2 < z < 8:", 2 < z < 8)

In [ ]:
# NumPy arrays refuse to be True or False as a whole
levels = np.array([1.0, 5.0])
try:
    if levels:
        print("never printed")
except ValueError as error:
    print("ValueError:", error)
print("levels.size > 0:", levels.size > 0)

### Idiom 10 - with: open, use, and always close.

`idioms/10_with.py`

```
MATLAB:  fid = fopen(...); c = onCleanup(@() fclose(fid)); ...
```

In [ ]:
import tempfile
import time
from contextlib import contextmanager
from pathlib import Path

import polars as pl

folder = Path(tempfile.mkdtemp())

In [ ]:
# A file: closed when the block ends, even if the block raises an error
with open(folder / "report.txt", "w") as file:
    file.write("M07: alarm\n")
print("closed after the block:", file.closed)

with open(folder / "report.txt") as file:
    print("read back:", file.read().strip())

In [ ]:
# The same pattern elsewhere: a setting that is restored afterwards
table = pl.DataFrame({"machine": ["M07"], "hours": [354.0]})
with pl.Config(ascii_tables=True, tbl_hide_dataframe_shape=True):
    print(table)

In [ ]:
# Writing your own is a function with a `yield` in the middle
@contextmanager
def timed(label):
    start = time.perf_counter()
    yield
    print(f"{label}: {(time.perf_counter() - start) * 1000:.1f} ms")


with timed("sum of a million numbers"):
    sum(range(1_000_000))

### Idiom 12 - Reading a class and a decorator.

`idioms/12_classes_and_decorators.py`

```
MATLAB:  classdef, with properties and methods
```

In [ ]:
import time
from dataclasses import dataclass

In [ ]:
# A class written out: __init__ builds the object, self is the object itself
class Machine:
    def __init__(self, name, hours=0.0):
        self.name = name
        self.hours = hours

    def is_old(self, limit=500):
        return self.hours > limit


m = Machine("M08", hours=582)
print("m.hours:", m.hours)
print("m.is_old():", m.is_old())
print("the same call, spelled out:", Machine.is_old(m))

In [ ]:
# The same class with @dataclass: __init__ and a readable print are written for you
@dataclass
class MachineRecord:
    name: str
    hours: float = 0.0

    def is_old(self, limit=500):
        return self.hours > limit


record = MachineRecord("M08", hours=582)
print(record)
print("record.is_old():", record.is_old())

In [ ]:
# A decorator is a function that takes a function and returns a new one
def timed(function):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = function(*args, **kwargs)
        print(f"{function.__name__} took {(time.perf_counter() - start) * 1000:.2f} ms")
        return result

    return wrapper


@timed
def total(n):
    return sum(range(n))


print(total(100_000))

In [ ]:
# Everything is an object: lists, strings and numbers have methods too
print("m07".upper(), [3, 1, 2].count(3), (2.5).is_integer())